# 05 — Silver: `ecommerce_itens_pedido`

**Squad 2 — Negócio em Tempo Real | Dupla José Vieira & Tiago Ramos**

Lê da Bronze só o que ainda não foi processado, aplica as regras da planilha e separa o resultado em três
destinos: **Silver** (válidos), **quarentena** (reprovados, com o motivo) e **espera** (itens cujo pedido
ainda não chegou à Silver de pedidos). Cada regra vira uma coluna `flag_*`.

T2 olha as duas saídas da Silver de pedidos: pedido aprovado libera o item; pedido **reprovado** (na quarentena
de pedidos) barra o item na hora; pedido que não está em nenhuma das duas ainda não chegou, e o item espera.

| Regra | Coluna | Critério | Destino |
|---|---|---|---|
| T1 | `flag_schema` | coluna obrigatória ausente no arquivo (100% nula no lote) ou nula na linha | quarentena |
| T2 | `flag_pedido_reprovado` | `id_pedido` está na quarentena de pedidos, e não na Silver | quarentena |
| T2 | `flag_sem_pedido` | `id_pedido` não está em nenhuma das duas | espera; quarentena após `ESPERA_MAX_H` |
| T3 | `flag_quantidade` | `quantidade` nula ou ≤ 0 | quarentena |
| T4 | `flag_preco` | `preco_unitario` ≤ 0, ou mais de 60% abaixo do `preco_lista` da Silver de produtos | quarentena |
| T5 | `flag_duplicado` | `id_item_pedido` repetido no lote ou já presente na Silver | quarentena |
| N9 | `flag_desconto` | `desconto_aplicado` > `preco_unitario` | quarentena |
| extra | `flag_desconto_negativo` | `desconto_aplicado` < 0 | quarentena |

A Silver leva também `dt_pedido` (da Silver de pedidos), `receita_bruta` e `receita_liquida` do item: a
Gold precisa da data para as janelas de tempo, e a tabela de itens não tem data própria.

In [ ]:
%run ./00_setup_config

## Parâmetros

`ESPERA_MAX_H`: quanto tempo um item pode aguardar o pedido. Pedido e itens chegam no mesmo lote do
gerador, então a espera cobre só a defasagem entre os notebooks; passado o prazo, o item é órfão.

Widget `reprocessar`: com `true`, ignora o que já foi processado, relê a Bronze inteira e **regrava** Silver,
quarentena e espera de itens (`overwrite`). Serve para aplicar uma regra nova ao histórico. O Delta guarda a
versão anterior, então dá para voltar atrás.

In [ ]:
from pyspark.sql import functions as F, Window

OBRIGATORIAS = ["id_item_pedido", "id_pedido", "sku", "quantidade", "preco_unitario", "desconto_aplicado"]
PRECO_MIN_CATALOGO, ESPERA_MAX_H = 0.4, 24   # T4: preco abaixo de 40% do preco_lista (mais de 60% menor) e erro
dbutils.widgets.text("reprocessar", "false")
REPROCESSAR = dbutils.widgets.get("reprocessar").strip().lower() == "true"
if REPROCESSAR:
    print("[reprocessar] Bronze inteira; Silver, quarentena e espera de itens serao regravadas")
print(f"Bronze: {TABELA_BRONZE}\nSilver: {CAMINHO_SILVER}\nQuarentena: {CAMINHO_QUARENTENA}\nEspera: {CAMINHO_ESPERA}")
print(f"Pedidos: {CAMINHO_SILVER_PEDIDOS}\nProdutos: {CAMINHO_SILVER_PRODUTOS}")

## Leitura incremental

O watermark é o maior `bronze_ingested_at` já visto na Silver, na quarentena **ou** na espera: sem olhar as
três, um lote inteiro barrado ou em espera seria reprocessado a cada execução. Os itens que estavam na
espera entram de novo no lote para nova tentativa.

In [ ]:
silver_atual, quar_atual, espera_atual = (None, None, None) if REPROCESSAR else (ler_delta(CAMINHO_SILVER), ler_delta(CAMINHO_QUARENTENA), ler_delta(CAMINHO_ESPERA))
marcas = [df.agg(F.max("bronze_ingested_at")).first()[0] for df in (silver_atual, quar_atual, espera_atual) if df is not None]
watermark = max([m for m in marcas if m is not None], default=None)

bronze = spark.table(TABELA_BRONZE).drop("ano", "mes", "dia", "hora")
novos = (bronze if watermark is None else bronze.filter(F.col("bronze_ingested_at") > watermark)).withColumn("espera_desde", F.current_timestamp())
lote = novos if espera_atual is None else novos.unionByName(espera_atual.select(*novos.columns))
n_novos, n_retentativa = novos.count(), (espera_atual.count() if espera_atual is not None else 0)
print(f"Watermark: {watermark} | novas na Bronze: {n_novos} | retentativas da espera: {n_retentativa}")

## Tabelas de apoio: pedidos e produtos

Sempre a **Silver** das outras tabelas, nunca o `raw`. Sem a Silver de pedidos, nenhum item pode ser
confirmado e todos vão para a espera. Sem a Silver de produtos, T4 checa só preço > 0 e o notebook avisa.

In [ ]:
pedidos_df, produtos_df, quar_pedidos_df = ler_delta(CAMINHO_SILVER_PEDIDOS), ler_delta(CAMINHO_SILVER_PRODUTOS), ler_delta(CAMINHO_QUARENTENA_PEDIDOS)
if pedidos_df is None:
    print("[aviso] Silver de pedidos nao encontrada: itens novos vao para a espera.")
    pedidos = spark.createDataFrame([], "id_pedido bigint, dt_pedido timestamp")
else:
    pedidos = pedidos_df.select(F.col("id_pedido").cast("bigint"), "dt_pedido").dropDuplicates(["id_pedido"])
if produtos_df is None:
    print("[aviso] Silver de produtos nao encontrada: T4 sem comparacao com o catalogo.")
    catalogo = spark.createDataFrame([], "sku string, preco_lista double")
else:
    catalogo = produtos_df.select("sku", F.col("preco_lista").cast("double")).dropDuplicates(["sku"])
if quar_pedidos_df is None:
    print("[aviso] quarentena de pedidos nao encontrada: item de pedido reprovado vai para a espera.")
    reprovados = spark.createDataFrame([], "id_pedido bigint")
else:  # pedido que voltou aprovado depois sai da lista
    reprovados = quar_pedidos_df.select(F.col("id_pedido").cast("bigint")).distinct().join(pedidos.select("id_pedido"), "id_pedido", "left_anti")
reprovados = reprovados.withColumn("_pedido_reprovado", F.lit(True))

## Regras e flags

T1: a Bronze lê com schema fixo, então coluna ausente no arquivo chega nula em **todas** as linhas daquele
arquivo; a checagem é "100% nula no `bronze_source_file`", além de nula na linha.
T5: dentro do lote fica a primeira ocorrência (Bronze mais antiga); as demais, e as que já estão na Silver,
vão para a quarentena como duplicadas, para a contagem de reenvios ficar visível.

In [ ]:
por_arquivo = Window.partitionBy("bronze_source_file")
primeiro = Window.partitionBy("id_item_pedido").orderBy("bronze_ingested_at", "bronze_source_file")
ja_na_silver = (silver_atual.select("id_item_pedido") if silver_atual is not None else spark.createDataFrame([], "id_item_pedido bigint")).withColumn("_na_silver", F.lit(True))

base = (lote.join(pedidos, "id_pedido", "left").join(reprovados, "id_pedido", "left").join(catalogo, "sku", "left").join(ja_na_silver, "id_item_pedido", "left")
            .withColumn("_rn", F.row_number().over(primeiro))
            .withColumn("_expirou", F.col("espera_desde") <= F.current_timestamp() - F.expr(f"INTERVAL {ESPERA_MAX_H} HOURS")))

regras = {
    "flag_schema":            F.greatest(*[F.max(F.col(c).isNotNull().cast("int")).over(por_arquivo) == 0 for c in OBRIGATORIAS], *[F.col(c).isNull() for c in OBRIGATORIAS]),
    "flag_quantidade":        F.col("quantidade").isNull() | (F.col("quantidade") <= 0),
    "flag_preco":             F.col("preco_unitario").isNull() | (F.col("preco_unitario") <= 0)
                              | F.coalesce((F.col("preco_lista") > 0) & (F.col("preco_unitario") < PRECO_MIN_CATALOGO * F.col("preco_lista")), F.lit(False)),
    "flag_duplicado":         (F.col("_rn") > 1) | F.coalesce(F.col("_na_silver"), F.lit(False)),
    "flag_desconto":          F.coalesce(F.col("desconto_aplicado") > F.col("preco_unitario"), F.lit(False)),
    "flag_desconto_negativo": F.coalesce(F.col("desconto_aplicado") < 0, F.lit(False)),
    "flag_pedido_reprovado":  F.col("dt_pedido").isNull() & F.coalesce(F.col("_pedido_reprovado"), F.lit(False)),
    "flag_sem_pedido":        F.col("dt_pedido").isNull() & ~F.coalesce(F.col("_pedido_reprovado"), F.lit(False)) & F.col("_expirou"),
}
validado = base.select("*", *[F.coalesce(c, F.lit(True)).alias(n) for n, c in regras.items()])
validado = validado.withColumn("quarantine_reason", F.concat_ws("; ", *[F.when(F.col(n), F.lit(n.removeprefix("flag_").upper())) for n in regras])) \
                   .withColumn("silver_processed_at", F.current_timestamp())

reprovado, sem_pai = F.col("quarantine_reason") != "", F.col("dt_pedido").isNull()
quarentena = validado.filter(reprovado).drop("_rn", "_na_silver", "_expirou", "_pedido_reprovado")
espera = validado.filter(~reprovado & sem_pai).select(*lote.columns)
validos = (validado.filter(~reprovado & ~sem_pai)
                   .withColumn("receita_bruta", F.round(F.col("preco_unitario") * F.col("quantidade"), 2))
                   .withColumn("receita_liquida", F.round((F.col("preco_unitario") - F.col("desconto_aplicado")) * F.col("quantidade"), 2))
                   .withColumn("ano", F.year("dt_pedido")).withColumn("mes", F.month("dt_pedido"))
                   .drop("_rn", "_na_silver", "_expirou", "_pedido_reprovado", "quarantine_reason", "preco_lista", "espera_desde", *regras))

## Gravação

**Silver:** `append`, particionada por ano/mês do `dt_pedido`; T5 garante que nenhum item entra duas vezes.
**Quarentena:** `append`, histórico de cada reprovação. **Espera:** `overwrite` com o que continua sem pedido.
A gravação é `overwrite` sobre a própria espera lida no início: o Delta lê uma versão fixa da tabela, então a
reescrita é segura. Com `reprocessar = true`, Silver e quarentena também são regravadas (`overwrite`).

In [ ]:
def gravar(df, caminho, modo, particao=None):
    w = df.write.format("delta").options(**adls_options).mode(modo).option("overwriteSchema" if modo == "overwrite" else "mergeSchema", "true")
    (w.partitionBy(*particao) if particao else w).save(caminho)


if n_novos + n_retentativa == 0:
    print("Nada novo na Bronze nem na espera; camadas inalteradas.")
else:
    n_val, n_quar, n_esp = validos.count(), quarentena.count(), espera.count()
    modo = "overwrite" if REPROCESSAR else "append"
    gravar(validos, CAMINHO_SILVER, modo, ["ano", "mes"])
    gravar(quarentena, CAMINHO_QUARENTENA, modo)
    if n_esp or espera_atual is not None or REPROCESSAR:
        gravar(espera, CAMINHO_ESPERA, "overwrite")
    print(f"Lote: {n_novos + n_retentativa} linhas | Silver +{n_val} | quarentena +{n_quar} | espera {n_esp}")

## Auditoria

In [ ]:
silver, quar, esp = ler_delta(CAMINHO_SILVER), ler_delta(CAMINHO_QUARENTENA), ler_delta(CAMINHO_ESPERA)
if silver is None:
    print("Silver ainda nao existe.")
else:
    n_silver = silver.count()
    print(f"Silver: {n_silver} linhas, {silver.select('id_item_pedido').distinct().count()} id_item_pedido distintos (devem ser iguais)")
    print(f"Espera: {esp.count() if esp is not None else 0} itens aguardando pedido")
    silver.printSchema()
if quar is not None:
    print(f"Quarentena acumulada: {quar.count()} linhas")
    display(quar.select([F.sum(F.col(n).cast("int")).alias(n) for n in regras]))
    display(quar.select("id_item_pedido", "id_pedido", "sku", "quantidade", "preco_unitario", "preco_lista", "desconto_aplicado", "quarantine_reason").limit(50))

## Insights: erros por regra

Quantas linhas violaram cada regra, sobre tudo o que já passou pela Silver (válidas + quarentena), e a
proporção de aprovadas e barradas. Uma linha pode violar mais de uma regra, então a soma das barras pode
passar do total da quarentena. Cores no padrão visual dos dashboards da squad.

In [ ]:
import matplotlib.pyplot as plt

AZUL, EIXO, FUNDO, OK, ERRO = "#1F4E8C", "#595959", "#F7F7F7", "#2E7D32", "#C0392B"
ROTULOS = {"flag_schema": "T1 schema", "flag_pedido_reprovado": "T2 pedido reprovado", "flag_sem_pedido": "T2 sem pedido", "flag_quantidade": "T3 quantidade ≤ 0", "flag_preco": "T4 preço / catálogo",
           "flag_duplicado": "T5 duplicado", "flag_desconto": "N9 desconto > preço", "flag_desconto_negativo": "Extra desconto < 0"}
if quar is None:
    print("Quarentena vazia: nenhuma regra violada ate agora.")
else:
    n_quar = quar.count(); total = n_quar + (n_silver if silver is not None else 0)
    erros = {n: quar.agg(F.sum(F.col(n).cast("int"))).first()[0] or 0 for n in ROTULOS}
    plt.rcParams.update({"font.family": "Arial", "axes.edgecolor": EIXO, "axes.labelcolor": EIXO, "xtick.color": EIXO, "ytick.color": EIXO, "text.color": "#262626"})
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={"width_ratios": [2, 1]}, facecolor=FUNDO)
    barras = ax1.barh(list(ROTULOS.values()), list(erros.values()), color=AZUL)
    ax1.bar_label(barras, labels=[f"{v}  ({v / total:.1%})" for v in erros.values()], padding=3)
    ax1.invert_yaxis(); ax1.set_xlim(0, max(max(erros.values()), 1) * 1.35); ax1.set_xlabel("linhas"); ax1.set_facecolor(FUNDO)
    ax1.spines[["top", "right"]].set_visible(False); ax1.set_title(f"Violações por regra ({total} linhas analisadas)", fontweight="bold")
    ax2.pie([total - n_quar, n_quar], colors=[OK, ERRO], autopct="%1.1f%%", startangle=90, wedgeprops={"width": 0.45}, pctdistance=0.78,
            textprops={"color": "white", "fontweight": "bold"})
    ax2.legend([f"Silver ({total - n_quar})", f"Quarentena ({n_quar})"], loc="lower center", bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False)
    ax2.set_title("Linhas aprovadas x barradas", fontweight="bold"); plt.tight_layout(); plt.show()

## Resumo

> **Base:** execução de 13/10/2026 sobre os 353 lotes de 21/09 a 08/10 (13.325 itens). Revisar após novas janelas.

**69% dos itens chegam limpos à Silver (9.196); 31% ficam na quarentena (4.129); nenhum em espera.**

- **Pedido reprovado (T2), 1.741 itens (13%):** o item é válido, mas o pedido dele foi barrado pelas regras
  de pedidos (valor, status, datas). Não é erro da tabela de itens: é a consequência, no detalhe, de um pedido
  que o negócio já não considera confiável. Nenhum item ficou órfão de verdade (`SEM_PEDIDO` = 0).
- **Preço fora do catálogo (T4), 1.531 itens (11,5%):** o maior problema próprio da tabela. Preço zero,
  negativo ou mais de 60% abaixo do `preco_lista` indica falha na busca do preço; se parte disso for
  promoção legítima, o limite de 60% precisa ser revisto com o negócio.
- **Desconto impossível (N9), 634 itens (4,8%):** desconto maior que o preço, item sairia com valor negativo.
- **Quantidade zero (T3), 469 itens (3,5%)**; 11 itens com desconto negativo.
- **Sem problema estrutural:** nenhuma coluna ausente (T1) e nenhum item reenviado (T5).

Para o negócio: quase um em cada três itens não pode entrar em receita nem em ranking de produtos sem
correção na origem. Os dois sinais mais fortes para levar ao time do sistema de vendas são o preço desalinhado
com o catálogo e o desconto acima do preço, que juntos somam cerca de 16% dos itens.